# Домашнє завдання: Рекомендаційні системи на реальних даних (Goodbooks-10k)

У цьому завданні Ви реалізуєте сучасні (advanced) архітектури рекомендаційних систем із фінального блоку лекції — але вже **не на іграшкових даних, а на реальному датасеті книжкових рейтингів Goodbooks-10k** (десятки тисяч користувачів, тисячі книг, мільйони оцінок).

Це дасть Вам змогу побачити, як підходи поводяться, коли даних справді багато: чому контентних ознак буває замало, як працює retrieval на тисячах елементів, і чому офлайн-метрики на кшталт Recall@K не такі високі, як хотілося б.

**Архітектури, які Ви зберете:** Vector Space Model, Two-Tower, Concat-based ranking (NCF) та двоетапний пайплайн Retrieval → Ranking.

**Стек:** `numpy`, `pandas`, `scikit-learn`, `torch`. GPU не обов'язковий, але з ним тренування буде швидшим (у Colab: *Runtime → Change runtime type → GPU*).

---

## Про датасет

[Goodbooks-10k](https://www.kaggle.com/datasets/zygmunt/goodbooks-10k) — це ~6 млн оцінок 10 000 найпопулярніших книг від 53 424 користувачів. Складається з кількох файлів:

- `ratings.csv` — оцінки: `user_id, book_id, rating` (1–5);
- `books.csv` — метадані книг: `book_id, goodreads_book_id, authors, title, average_rating, ...`;
- `book_tags.csv` — теги/полиці, які користувачі вішали на книги: `goodreads_book_id, tag_id, count`;
- `tags.csv` — розшифровка тегів: `tag_id, tag_name`.

**Важливий нюанс:** на відміну від навчального прикладу, тут **немає готових жанрів**. Жанри доведеться сконструювати самостійно з користувацьких тегів — а це шумні дані (юзери можуть зазначати що завгодно). Це реалістична задача feature engineering, і ми її розберемо в підготовчій частині.

Ще один нюанс із реальних даних: `book_tags.csv` посилається на `goodreads_book_id`, а `ratings.csv` — на `book_id`. Щоб їх поєднати, потрібен джойн через `books.csv`.


## Крок 0. Завантаження даних

Є три способи дістати дані — оберіть будь-який.

**Спосіб A — Kaggle API (рекомендований).** Завантаження з Kaggle API. Зручно, бо декілька файлів і вони завантажаться всі самостійно. Для цього способу завантажте свій `kaggle.json` (Kaggle → Account → Create New API Token), потім виконайте:
```python
from google.colab import files; files.upload()   # оберіть kaggle.json
```
і розкоментуйте відповідний блок нижче.

**Спосіб B — ручне завантаження.** Завантажте архів з посилання на датасет вище з Kaggle, розпакуйте і покладіть `ratings.csv`, `books.csv`, `book_tags.csv`, `tags.csv` поруч із ноутбуком (або через панель Files у Colab).

**Спосіб C — GitHub-дзеркало (фолбек).** Оригінальний автор виклав файли і на GitHub — код нижче підхопить їх автоматично, якщо локально файлів немає.


In [ ]:
# (Спосіб A) Kaggle API — розкоментуйте, якщо завантажили kaggle.json
# !pip -q install kaggle
# import os, shutil
# os.makedirs("/root/.kaggle", exist_ok=True)
# shutil.move("kaggle.json", "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
# !kaggle datasets download -d zygmunt/goodbooks-10k --unzip -p .

In [4]:
import os
import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master"
FILES = ["ratings.csv", "books.csv", "book_tags.csv", "tags.csv"]

def load(fname):
    """Спочатку шукаємо файл локально, інакше тягнемо з GitHub-дзеркала."""
    if os.path.exists(fname):
        return pd.read_csv(fname)
    print(f"{fname} не знайдено локально — завантажую з GitHub...")
    return pd.read_csv(f"{GITHUB}/{fname}")

ratings = load("ratings.csv")
books = load("books.csv")
book_tags = load("book_tags.csv")
tags = load("tags.csv")

print("ratings:", ratings.shape)
print("books:  ", books.shape)
print("book_tags:", book_tags.shape, "| tags:", tags.shape)
books[["book_id", "authors", "title", "average_rating"]].head()

ratings.csv не знайдено локально — завантажую з GitHub...
books.csv не знайдено локально — завантажую з GitHub...
book_tags.csv не знайдено локально — завантажую з GitHub...
tags.csv не знайдено локально — завантажую з GitHub...
ratings: (5976479, 3)
books:   (10000, 23)
book_tags: (999912, 3) | tags: (34252, 2)


,book_id,authors,title,average_rating
0,1,Suzanne Collins,"The Hunger Games (The Hunger Games, #1)",4.34
1,2,"J.K. Rowling, Mary GrandPré",Harry Potter and the Sorcerer's Stone (Harry P...,4.44
2,3,Stephenie Meyer,"Twilight (Twilight, #1)",3.57
3,4,Harper Lee,To Kill a Mockingbird,4.25
4,5,F. Scott Fitzgerald,The Great Gatsby,3.89


## Крок 1. Інженерія жанрів із тегів (feature engineering)

Жанрів у датасеті немає, але є користувацькі теги. Виберемо набір канонічних жанрів і для кожної книги позначимо, які з них їй приписали користувачі. Так ми отримаємо **бінарну матрицю book × genre** — це й будуть контентні ознаки айтемів (аналог `movie_feats_df` із лекції, але здобутий з реальних шумних даних).


In [5]:
# Канонічні жанри, які шукаємо серед тегів
GENRES = ["fantasy", "romance", "mystery", "thriller", "horror", "historical",
          "science-fiction", "young-adult", "nonfiction", "classics",
          "contemporary", "crime"]

# tag_name -> tag_id
name_to_tagid = dict(zip(tags["tag_name"], tags["tag_id"]))
genre_tag_ids = {g: name_to_tagid[g] for g in GENRES if g in name_to_tagid}

# book_tags використовує goodreads_book_id -> мапимо у book_id через books.csv
gid_to_bid = dict(zip(books["goodreads_book_id"], books["book_id"]))
tagid_to_genre = {tid: g for g, tid in genre_tag_ids.items()}

bt = book_tags[book_tags["tag_id"].isin(genre_tag_ids.values())].copy()
bt["book_id"] = bt["goodreads_book_id"].map(gid_to_bid)
bt = bt.dropna(subset=["book_id"])
bt["genre"] = bt["tag_id"].map(tagid_to_genre)

# бінарна матриця book × genre (жанр присутній, якщо користувачі його тегали)
genre_matrix = (
    bt.pivot_table(index="book_id", columns="genre", values="count", aggfunc="sum", fill_value=0)
      .reindex(columns=GENRES, fill_value=0) > 0
).astype(int)

print("Книг із хоча б одним жанром:", (genre_matrix.sum(axis=1) > 0).sum(), "/", len(books))
print("\nРозподіл жанрів:")
print(genre_matrix.sum().sort_values(ascending=False))
genre_matrix.head()

Книг із хоча б одним жанром: 9954 / 10000

Розподіл жанрів:
genre
contemporary       5287
fantasy            4259
romance            4251
mystery            3686
young-adult        3630
classics           2785
historical         2544
thriller           2522
science-fiction    2222
crime              2083
nonfiction         1833
horror             1372
dtype: int64


genre,fantasy,romance,mystery,thriller,horror,historical,science-fiction,young-adult,nonfiction,classics,contemporary,crime
book_id,,,,,,,,,,,,
1,1,1,0,1,0,0,1,1,0,0,1,0
2,1,0,1,0,0,0,0,1,0,1,1,0
3,1,0,0,0,1,0,1,1,0,0,1,0
4,0,0,1,0,0,1,0,1,0,1,1,1
5,0,1,0,0,0,1,0,1,0,1,0,0


## Крок 2. Підвибірка під Colab

6 млн рейтингів — забагато для навчального ноутбука на CPU. Візьмемо **топ-N найпопулярніших книг** і **активних користувачів** (хто поставив ≥ 20 оцінок), а тоді обмежимо число користувачів. Так зберігається щільність взаємодій, а тренування лишається швидким.

> Якщо у Вас GPU або багато часу — сміливо збільшуйте `TOP_BOOKS` та `N_USERS`.


In [6]:
TOP_BOOKS = 1500       # скільки найпопулярніших книг лишити
MIN_USER_RATINGS = 20  # мінімум оцінок на користувача
N_USERS = 2000         # скільки користувачів узяти у підвибірку
LIKE_THRESHOLD = 4     # rating >= 4 вважаємо "лайком" (позитивна взаємодія)

rng = np.random.RandomState(42)

top_books = ratings["book_id"].value_counts().head(TOP_BOOKS).index
r = ratings[ratings["book_id"].isin(top_books)]
active = r["user_id"].value_counts()
r = r[r["user_id"].isin(active[active >= MIN_USER_RATINGS].index)]
sample_users = rng.choice(r["user_id"].unique(), size=min(N_USERS, r["user_id"].nunique()), replace=False)
r = r[r["user_id"].isin(sample_users)].copy()

# лишаємо тільки книги, для яких є жанрові ознаки
r = r[r["book_id"].isin(genre_matrix.index)].copy()

items = sorted(r["book_id"].unique())
users = sorted(r["user_id"].unique())
genre_matrix = genre_matrix.reindex(items).fillna(0).astype(int)

print(f"Взаємодій: {len(r):,} | користувачів: {len(users):,} | книг: {len(items):,}")
print(f"Щільність: {len(r) / (len(users) * len(items)):.4f}")

Взаємодій: 140,934 | користувачів: 2,000 | книг: 1,496
Щільність: 0.0471


In [7]:
import torch
import torch.nn as nn

torch.manual_seed(42)

user_to_idx = {u: i for i, u in enumerate(users)}
item_to_idx = {b: i for i, b in enumerate(items)}
title_of = dict(zip(books["book_id"], books["title"]))

item_feats = torch.tensor(genre_matrix.values, dtype=torch.float32)  # (M, n_genres)
M = len(items)
n_genres = item_feats.shape[1]

# train/val split по взаємодіях
r = r.sample(frac=1, random_state=42).reset_index(drop=True)
n_val = int(len(r) * 0.2)
val_df = r.iloc[:n_val]
train_df = r.iloc[n_val:]

# позитивні пари (лайки) у train
train_pos = train_df[train_df["rating"] >= LIKE_THRESHOLD]
pos_u = torch.tensor([user_to_idx[u] for u in train_pos["user_id"]])
pos_i = torch.tensor([item_to_idx[b] for b in train_pos["book_id"]])

# що користувач уже бачив (щоб не рекомендувати повторно і не семплити як негатив)
from collections import defaultdict
seen_by_user = defaultdict(set)
for u, b in zip(train_df["user_id"], train_df["book_id"]):
    seen_by_user[user_to_idx[u]].add(item_to_idx[b])

# val-лайки для оцінки якості
val_pos = defaultdict(set)
for row in val_df.itertuples():
    if row.rating >= LIKE_THRESHOLD:
        val_pos[user_to_idx[row.user_id]].add(item_to_idx[row.book_id])

print(f"Позитивних пар у train: {len(pos_u):,} | користувачів з val-лайками: {len(val_pos):,}")

Позитивних пар у train: 77,070 | користувачів з val-лайками: 1,991


## Крок 3. Метрика оцінки якості рангування

В лекції ми з вами для оцінки якості використовували **RMSE**. Це валідний варіант, коли треба швидко оцінити якість рек. моделі, але спрощений. RMSE показує, наскільки точно модель передбачає оцінку, яку користувач поставить елементу.

В реальних системах нас ще цікавить **якість ранжування** — наскільки релевантні елементи потрапили в топ списку, який ми реально показуємо користувачу. Для цього використовують ранжувальні метрики: **Precision@K**, **Recall@K**, **NDCG**, **MAP**, **MRR**.

Детальніше можна познайомитись з цими мериками тут:
- огляд метрик для рекомендаційних систем: https://www.evidentlyai.com/ranking-metrics/evaluating-recommender-systems
- Precision та Recall at K: https://www.evidentlyai.com/ranking-metrics/precision-recall-at-k

Нижче давайте реалізуємо функцію `recall_at_k` і будемо оцінювати нею всі наші моделі.

![](https://cdn.prod.website-files.com/660ef16a9e0687d9cc27474a/662c4327f27ee08d3e4d4b2e_6577812c4d677925f1ab5f84_precision_recall_k9.png)

![](https://cdn.prod.website-files.com/660ef16a9e0687d9cc27474a/662c4327f27ee08d3e4d4b47_657781b1f9c868e0cda088f6_precision_recall_k11.png)

**Як працює `recall_at_k`:**

1. Для кожного користувача ми беремо його реальні вподобання з валідаційної вибірки (`val_pos` — книги, які він оцінив на ≥ 4), просимо модель оцінити всі книги й відбираємо топ-K рекомендацій. Перед цим прибираємо книги, які користувач уже бачив у train (щоб не рекомендувати відоме).

2. Далі рахуємо, скільки книг із топ-K справді потрапили в його вподобання (`hits`), і ділимо на загальну кількість релевантних книг (обмежену K, бо більше за K у топ і не влізе).

3. Усереднюємо по всіх користувачах — і отримуємо одне число від 0 до 1: **яку частку того, що користувачу реально сподобалось, модель змогла підняти в топ-K.**

In [8]:
def recall_at_k(score_fn, k=10):
    """Частка val-лайків, що потрапили у топ-k рекомендацій (усереднена по користувачах).
    score_fn(user_idx_tensor) -> матриця оцінок (n_users, M)."""
    eval_users = list(val_pos.keys())
    hits, total = 0, 0
    with torch.no_grad():
        scores = score_fn(torch.tensor(eval_users))  # (len(eval_users), M)
        for row, u in enumerate(eval_users):
            s = scores[row].clone()
            for i in seen_by_user[u]:
                s[i] = -1e9  # прибираємо вже побачене
            topk = torch.topk(s, k).indices.tolist()
            truth = val_pos[u]
            hits += len(set(topk) & truth)
            total += min(len(truth), k)
    return hits / max(total, 1)

---
## Завдання 1. Vector Space Model (векторний підхід)

Перетворимо і книги, і користувачів на вектори в спільному просторі та шукатимемо рекомендації через cosine similarity. Роль ембединга книги відіграє її **нормалізований вектор жанрів** (пояснення про нормалізацію - нижче), а вектор користувача збираємо як **average pooling** ембедингів книг, які він уподобав.

**Що зробити:**

1. Побудуйте `item_emb` — матрицю L2-нормалізованих жанрових векторів усіх книг.
2. Реалізуйте функцію `user_vector(user_idx)` — зважене (за оцінкою) середнє ембедингів уподобаних книг користувача.
3. Реалізуйте функцію `vsm_scores(user_idxs)` — оцінки (cosine) усіх книг для набору користувачів, та порахуйте `recall_at_k`.
4. Покажіть топ-5 рекомендацій для одного користувача (з назвами книг).

**Довідка:**

L2-нормалізація — це ділення вектора на його довжину (L2-норму), щоб отримати вектор тієї ж напрямленості, але одиничної довжини.

Норма рахується як корінь із суми квадратів компонент:

$$\|v\|_2 = \sqrt{(v_1^2 + v_2^2 + \dots + v_n^2)}$$

а сам нормалізований вектор — це
$$\hat{v} = \frac{v}{\|v\|_2}$$

Навіщо це в рекомендаційних системах: після нормалізації **косинусна подібність зводиться до простого скалярного добутку**. Бо $\cos(a, b) = \frac{a \cdot b}{\|a\|\|b\|}$, і якщо обидва вектори вже одиничної довжини, знаменник = 1, тож $\cos(a,b) = a \cdot b$. Це і швидше, і прибирає вплив «довжини» вектора — порівнюється лише напрямок (тобто склад жанрів/смаків), а не те, скільки книг користувач оцінив.

*Приклад:*

Вектор `[3, 4]` має довжину $\sqrt{(9+16)}=5$, після нормалізації стає `[0.6, 0.8]` — напрямок той самий, довжина 1.

In [9]:
import torch.nn.functional as F

item_emb = F.normalize(item_feats, p=2, dim=1)

print(item_emb.shape)

torch.Size([1496, 12])


In [10]:
# train-лайки з рейтингами
train_likes = train_df[train_df["rating"] >= LIKE_THRESHOLD].copy()

liked_by_user = defaultdict(list)

for row in train_likes.itertuples():
    u_idx = user_to_idx[row.user_id]
    i_idx = item_to_idx[row.book_id]
    rating = row.rating
    liked_by_user[u_idx].append((i_idx, rating))

In [56]:
# 2. Вектор користувача як зважене середнє ембедингів уподобаних книг
def user_vector(user_idx):
    liked_items = liked_by_user[user_idx]

    if len(liked_items) == 0:
        return torch.zeros(n_genres)

    item_ids = torch.tensor([i for i, rating in liked_items])
    weights = torch.tensor([rating for i, rating in liked_items], dtype=torch.float32)

    emb = item_emb[item_ids]  # (n_liked, n_genres)

    # зважене середнє
    user_emb = (emb * weights.unsqueeze(1)).sum(dim=0) / weights.sum()

    # нормалізуємо профіль користувача
    user_emb = F.normalize(user_emb.unsqueeze(0), p=2, dim=1).squeeze(0)

    return user_emb

In [57]:
# 3. Оцінки всіх книг для набору користувачів
def vsm_scores(user_idxs):
    user_vecs = torch.stack([
        user_vector(int(u)) for u in user_idxs
    ])

    # cosine similarity, бо item_emb і user_vecs нормалізовані
    scores = user_vecs @ item_emb.T

    return scores

In [58]:
recall = recall_at_k(vsm_scores, k=10)
print(f"VSM Recall@10: {recall:.4f}")

VSM Recall@10: 0.0515


In [59]:
def recommend_vsm(user_idx, k=5):
    with torch.no_grad():
        scores = vsm_scores(torch.tensor([user_idx]))[0]

        # не рекомендуємо вже бачені книги
        for i in seen_by_user[user_idx]:
            scores[i] = -1e9

        top_items = torch.topk(scores, k).indices.tolist()

    recs = []
    for i in top_items:
        book_id = items[i]
        recs.append({
            "book_id": book_id,
            "title": title_of.get(book_id, "Unknown"),
            "score": float(scores[i])
        })

    return recs

In [60]:
example_user = users[0]
example_user_idx = user_to_idx[example_user]

recommend_vsm(example_user_idx, k=5)

[{'book_id': np.int64(496),
  'title': "The Clan of the Cave Bear (Earth's Children, #1)",
  'score': 0.8850772976875305},
 {'book_id': np.int64(377), 'title': 'Stardust', 'score': 0.8850772976875305},
 {'book_id': np.int64(35),
  'title': 'The Alchemist',
  'score': 0.8717203736305237},
 {'book_id': np.int64(314),
  'title': 'Inkheart (Inkworld, #1)',
  'score': 0.8680949807167053},
 {'book_id': np.int64(635),
  'title': "Sophie's World",
  'score': 0.865156888961792}]

**Питання:** Recall@10 у векторного підходу досить низький. Чому?


- Усі книги описуються тільки 12-ма жанрами, тобто дві книги можуть мати одитнаковий вектор, але бути різними для читача, оскільки не враховано "піджанр"
- немає collaborative information, тобто не враховує того факту, що користувачі, які люблять книгу А, часто люблять також книгу Б
- ми будуємо профіль користувача як середнє всіх вподобаних книг, тобто якщо користувач уподобав трилер і фентезі - його вектор стає чимось середнім між ними. В результаті модель може ректомендувати книги, які не є ні хорошим фентезі ні хорошим трилером.

---
## Завдання 2. Two-Tower архітектура

У Завданні 1 вектор користувача рахувався «вручну». Two-Tower натомість **навчає дві окремі башти**: User Tower (з ембединга user_id) та Item Tower (з жанрових ознак). Мережа зводить вектори уподобаних пар близько, а випадкових — далеко. Перевага: вектори книг рахуються один раз і кладуться в індекс (наприклад, FAISS) для швидкого retrieval — рахувати в реальному часі треба лише вектор користувача. Це **late fusion**.

**Що зробити:**

1. Реалізуйте `TwoTower` (user_tower через `nn.Embedding`, item_tower зі жанрових ознак), виходи L2-нормалізуйте.
2. Навчіть на лайках як позитивах і **negative sampling з усього корпусу** (як у пейпері від YouTube) з `BCEWithLogitsLoss` - він є реалізований в PyTorch.
3. Порахуйте `recall_at_k` через попередньо обчислені вектори книг і покажіть приклад рекомендацій.

> **Підказка.** Множте логіти на «температуру» (\~10), бо скалярний добуток нормалізованих векторів лежить у [-1, 1].
> Множення на температуру (\~10) розтягує діапазон логітів до [-10, 10], і тоді сигмоїда може видавати по-справжньому впевнені ймовірності (близькі до 0 і 1). Це дає лосу нормальний градієнт і модель навчається.


In [61]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TwoTower(nn.Module):
    def __init__(self, n_users, n_genres, emb_dim=32, hidden_dim=64, temperature=10.0):
        super().__init__()
        self.temperature = temperature

        self.user_tower = nn.Embedding(n_users, emb_dim)

        self.item_tower = nn.Sequential(
            nn.Linear(n_genres, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, emb_dim)
        )

    def encode_users(self, user_idxs):
        u = self.user_tower(user_idxs)
        return F.normalize(u, p=2, dim=1)

    def encode_items(self, item_feats_batch):
        i = self.item_tower(item_feats_batch)
        return F.normalize(i, p=2, dim=1)

    def forward(self, user_idxs, item_feats_batch):
        u = self.encode_users(user_idxs)
        i = self.encode_items(item_feats_batch)

        logits = (u * i).sum(dim=1) * self.temperature
        return logits

In [62]:
def two_tower_scores(user_idxs):
    model.eval()

    user_idxs = user_idxs.to(device)

    with torch.no_grad():
        user_emb = model.encode_users(user_idxs)
        item_emb = model.encode_items(item_feats_device)

        scores = user_emb @ item_emb.T

    return scores.cpu()

In [63]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = TwoTower(
    n_users=len(users),
    n_genres=n_genres,
    emb_dim=32,
    hidden_dim=64,
    temperature=10.0
).to(device)

item_feats_device = item_feats.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

BATCH_SIZE = 1024
EPOCHS = 10
NEG_PER_POS = 1

pos_u_device = pos_u.to(device)
pos_i_device = pos_i.to(device)

for epoch in range(EPOCHS):
    model.train()

    perm = torch.randperm(len(pos_u_device))
    total_loss = 0

    for start in range(0, len(pos_u_device), BATCH_SIZE):
        idx = perm[start:start + BATCH_SIZE]

        batch_u_pos = pos_u_device[idx]
        batch_i_pos = pos_i_device[idx]

        # positive examples
        pos_labels = torch.ones(len(batch_u_pos), device=device)

        # negative sampling з усього корпусу
        batch_u_neg = batch_u_pos.repeat_interleave(NEG_PER_POS)
        batch_i_neg = torch.randint(
            low=0,
            high=M,
            size=(len(batch_u_neg),),
            device=device
        )
        neg_labels = torch.zeros(len(batch_u_neg), device=device)

        # об'єднуємо positive + negative
        batch_u = torch.cat([batch_u_pos, batch_u_neg])
        batch_i = torch.cat([batch_i_pos, batch_i_neg])
        labels = torch.cat([pos_labels, neg_labels])

        logits = model(batch_u, item_feats_device[batch_i])
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / max(1, len(pos_u_device) // BATCH_SIZE)
    recall = recall_at_k(lambda u: two_tower_scores(u), k=10)

    print(f"Epoch {epoch+1}: loss={avg_loss:.4f}, Recall@10={recall:.4f}")

Epoch 1: loss=0.9593, Recall@10=0.0110
Epoch 2: loss=0.8830, Recall@10=0.0109
Epoch 3: loss=0.8224, Recall@10=0.0112
Epoch 4: loss=0.7784, Recall@10=0.0125
Epoch 5: loss=0.7475, Recall@10=0.0133
Epoch 6: loss=0.7281, Recall@10=0.0133
Epoch 7: loss=0.7159, Recall@10=0.0145
Epoch 8: loss=0.7073, Recall@10=0.0140
Epoch 9: loss=0.7022, Recall@10=0.0154
Epoch 10: loss=0.6980, Recall@10=0.0161


In [64]:
def recommend_two_tower(user_idx, k=5):
    model.eval()

    with torch.no_grad():
        scores = two_tower_scores(torch.tensor([user_idx]))[0]

        for i in seen_by_user[user_idx]:
            scores[i] = -1e9

        top_items = torch.topk(scores, k).indices.tolist()

    recs = []

    for i in top_items:
        book_id = items[i]
        recs.append({
            "book_id": book_id,
            "title": title_of.get(book_id, "Unknown"),
            "score": float(scores[i])
        })

    return recs

In [65]:
example_user = users[0]
example_user_idx = user_to_idx[example_user]

for rec in recommend_two_tower(example_user_idx, k=5):
    print(f"{rec['title']} | score={rec['score']:.4f}")

The Perfect Storm: A True Story of Men Against the Sea | score=0.0645
The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics | score=0.0600
Unbroken: A World War II Story of Survival, Resilience, and Redemption | score=0.0600
The Audacity of Hope: Thoughts on Reclaiming the American Dream | score=0.0600
The Art of War | score=0.0600


---
## Завдання 3. Concat-based ranking (NCF)

На відміну від Two-Tower (late fusion), тут **early fusion**: склеюємо ембединг користувача і ознаки книги в один вектор і пропускаємо через MLP, який сам моделює крос-взаємодії. Платою є те, що модель **не можна заіндексувати** — щоб знайти найкращу книгу, треба прогнати кожну пару (user, item). Тому її використовують лише на фінальному ранжуванні кількох кандидатів.

**Що зробити:**

1. Реалізуйте `NCF`: `concat(user_embedding, item_genre_features)` → MLP → один логіт.
2. Навчіть на тих самих позитивах/негативах.
3. Реалізуйте `rank_ncf(user_idx, candidate_idxs)` — ранжування заданого списку кандидатів за `sigmoid` логіта.


In [66]:
class NCF(nn.Module):
    def __init__(self, n_users, n_genres, emb_dim=32, hidden_dim=64):
        super().__init__()

        self.user_emb = nn.Embedding(n_users, emb_dim)

        self.mlp = nn.Sequential(
            nn.Linear(emb_dim + n_genres, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, user_idxs, item_feats_batch):
        u = self.user_emb(user_idxs)
        x = torch.cat([u, item_feats_batch], dim=1)
        logits = self.mlp(x).squeeze(1)
        return logits

In [67]:
ncf = NCF(
    n_users=len(users),
    n_genres=n_genres,
    emb_dim=32,
    hidden_dim=64
).to(device)

optimizer = torch.optim.Adam(ncf.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

BATCH_SIZE = 1024
EPOCHS = 10
NEG_PER_POS = 1

for epoch in range(EPOCHS):
    ncf.train()

    perm = torch.randperm(len(pos_u_device))
    total_loss = 0
    n_batches = 0

    for start in range(0, len(pos_u_device), BATCH_SIZE):
        idx = perm[start:start + BATCH_SIZE]

        batch_u_pos = pos_u_device[idx]
        batch_i_pos = pos_i_device[idx]

        pos_labels = torch.ones(len(batch_u_pos), device=device)

        # negative sampling
        batch_u_neg = batch_u_pos.repeat_interleave(NEG_PER_POS)
        batch_i_neg = torch.randint(
            low=0,
            high=M,
            size=(len(batch_u_neg),),
            device=device
        )
        neg_labels = torch.zeros(len(batch_u_neg), device=device)

        batch_u = torch.cat([batch_u_pos, batch_u_neg])
        batch_i = torch.cat([batch_i_pos, batch_i_neg])
        labels = torch.cat([pos_labels, neg_labels])

        logits = ncf(batch_u, item_feats_device[batch_i])
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    print(f"Epoch {epoch+1}: loss={total_loss / n_batches:.4f}")

Epoch 1: loss=0.6861
Epoch 2: loss=0.6789
Epoch 3: loss=0.6746
Epoch 4: loss=0.6692
Epoch 5: loss=0.6615
Epoch 6: loss=0.6536
Epoch 7: loss=0.6433
Epoch 8: loss=0.6329
Epoch 9: loss=0.6240
Epoch 10: loss=0.6164


In [68]:
def rank_ncf(user_idx, candidate_idxs):
    ncf.eval()

    user_tensor = torch.tensor(
        [user_idx] * len(candidate_idxs),
        dtype=torch.long,
        device=device
    )

    item_tensor = torch.tensor(
        candidate_idxs,
        dtype=torch.long,
        device=device
    )

    with torch.no_grad():
        logits = ncf(user_tensor, item_feats_device[item_tensor])
        scores = torch.sigmoid(logits)

    order = torch.argsort(scores, descending=True)

    ranked = []

    for idx in order.cpu().tolist():
        item_idx = candidate_idxs[idx]
        book_id = items[item_idx]

        ranked.append({
            "item_idx": item_idx,
            "book_id": book_id,
            "title": title_of.get(book_id, "Unknown"),
            "score": float(scores[idx].cpu())
        })

    return ranked

In [69]:
example_user_idx = user_to_idx[users[0]]

# кандидати: всі книги, які користувач ще не бачив
candidate_idxs = [
    i for i in range(M)
    if i not in seen_by_user[example_user_idx]
]

top5 = rank_ncf(example_user_idx, candidate_idxs)[:5]

for rec in top5:
    print(f"{rec['title']} | score={rec['score']:.4f}")

Pride and Prejudice and Zombies (Pride and Prejudice and Zombies, #1) | score=0.8284
The Clan of the Cave Bear (Earth's Children, #1) | score=0.8215
Stardust | score=0.8215
The Vampire Lestat (The Vampire Chronicles, #2) | score=0.7556
The Queen of the Damned (The Vampire Chronicles, #3) | score=0.7556


In [70]:
def ncf_scores(user_idxs):
    ncf.eval()

    all_scores = []

    with torch.no_grad():
        for u in user_idxs.tolist():
            user_tensor = torch.full(
                (M,),
                int(u),
                dtype=torch.long,
                device=device
            )

            logits = ncf(user_tensor, item_feats_device)
            scores = torch.sigmoid(logits)

            all_scores.append(scores.cpu())

    return torch.stack(all_scores)

In [71]:
recall_ncf = recall_at_k(ncf_scores, k=10)
print(f"NCF Recall@10: {recall_ncf:.4f}")

NCF Recall@10: 0.0383


---
## Завдання 4. Двоетапний пайплайн Retrieval → Ranking

Поєднаємо все так, як це працює у великих системах: **Two-Tower швидко відбирає кандидатів** (retrieval серед усіх книг), а **NCF точно ранжує** цю коротку добірку.

**Що зробити:**

1. `retrieve(user_idx, n_candidates)` — топ-N книг за Two-Tower (Завдання 2), без уже побачених.
2. `recommend_pipeline(user_idx, n_candidates, top_k)` — прогнати кандидатів через `rank_ncf` (Завдання 3).
3. Показати для кількох користувачів: що відібрав retrieval і що залишив ranking.


In [72]:
def retrieve(user_idx, n_candidates=100):
    model.eval()

    with torch.no_grad():
        scores = two_tower_scores(torch.tensor([user_idx]))[0]

        # прибираємо вже побачені книги
        for i in seen_by_user[user_idx]:
            scores[i] = -1e9

        candidate_idxs = torch.topk(scores, n_candidates).indices.tolist()

    return candidate_idxs

In [73]:
def recommend_pipeline(user_idx, n_candidates=100, top_k=10):
    candidate_idxs = retrieve(user_idx, n_candidates=n_candidates)

    ranked = rank_ncf(user_idx, candidate_idxs)

    return ranked[:top_k]

In [74]:
def show_retrieval(user_idx, n_candidates=10):
    candidate_idxs = retrieve(user_idx, n_candidates=n_candidates)

    print(f"Retrieval candidates for user_idx={user_idx}:")
    for i in candidate_idxs:
        book_id = items[i]
        print(f"- {title_of.get(book_id, 'Unknown')}")

In [75]:
def show_pipeline(user_idx, n_candidates=100, top_k=10):
    candidate_idxs = retrieve(user_idx, n_candidates=n_candidates)
    ranked = rank_ncf(user_idx, candidate_idxs)[:top_k]

    print(f"\nUser idx: {user_idx}")
    print("\nRetrieval top candidates:")
    for i in candidate_idxs[:top_k]:
        book_id = items[i]
        print(f"- {title_of.get(book_id, 'Unknown')}")

    print("\nFinal NCF ranking:")
    for rec in ranked:
        print(f"- {rec['title']} | score={rec['score']:.4f}")

In [76]:
for user_idx in [0, 1, 2]:
    show_pipeline(
        user_idx=user_idx,
        n_candidates=100,
        top_k=5
    )


User idx: 0

Retrieval top candidates:
- The Perfect Storm: A True Story of Men Against the Sea
- In the Garden of Beasts: Love, Terror, and an American Family in Hitler's Berlin
- The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics
- Dead Wake: The Last Crossing of the Lusitania
- The Audacity of Hope: Thoughts on Reclaiming the American Dream

Final NCF ranking:
- I Am Malala: The Story of the Girl Who Stood Up for Education and Was Shot by the Taliban | score=0.6612
- A Long Way Gone: Memoirs of a Boy Soldier | score=0.6612
- Wild: From Lost to Found on the Pacific Crest Trail | score=0.5359
- Born to Run: A Hidden Tribe, Superathletes, and the Greatest Race the World Has Never Seen | score=0.5359
- Sh*t My Dad Says | score=0.5359

User idx: 1

Retrieval top candidates:
- Leaves of Grass
- A Brief History of Time
- The Complete Poems of Emily Dickinson
- The Prophet
- The Man Who Mistook His Wife for a Hat and Other Clinical Tales

Final N

In [77]:
def pipeline_scores(user_idxs, n_candidates=100):
    all_scores = torch.full((len(user_idxs), M), -1e9)

    for row, u in enumerate(user_idxs.tolist()):
        candidate_idxs = retrieve(int(u), n_candidates=n_candidates)
        ranked = rank_ncf(int(u), candidate_idxs)

        for rec in ranked:
            all_scores[row, rec["item_idx"]] = rec["score"]

    return all_scores

In [78]:
recall_pipeline = recall_at_k(
    lambda u: pipeline_scores(u, n_candidates=100),
    k=10
)

print(f"Pipeline Recall@10: {recall_pipeline:.4f}")

Pipeline Recall@10: 0.0283


In [79]:
def show_pipeline_with_positions(user_idx, n_candidates=100, top_k=5):
    candidate_idxs = retrieve(user_idx, n_candidates=n_candidates)
    ranked = rank_ncf(user_idx, candidate_idxs)[:top_k]

    retrieval_rank = {
        item_idx: rank + 1
        for rank, item_idx in enumerate(candidate_idxs)
    }

    print(f"\nUser idx: {user_idx}")

    print("\nFinal NCF ranking:")
    for new_rank, rec in enumerate(ranked, start=1):
        old_rank = retrieval_rank[rec["item_idx"]]
        print(
            f"{new_rank}. {rec['title']} "
            f"| NCF score={rec['score']:.4f} "
            f"| retrieval rank={old_rank}"
        )

In [80]:
show_pipeline_with_positions(0)


User idx: 0

Final NCF ranking:
1. I Am Malala: The Story of the Girl Who Stood Up for Education and Was Shot by the Taliban | NCF score=0.6612 | retrieval rank=81
2. A Long Way Gone: Memoirs of a Boy Soldier | NCF score=0.6612 | retrieval rank=80
3. Wild: From Lost to Found on the Pacific Crest Trail | NCF score=0.5359 | retrieval rank=41
4. Born to Run: A Hidden Tribe, Superathletes, and the Greatest Race the World Has Never Seen | NCF score=0.5359 | retrieval rank=33
5. Sh*t My Dad Says | NCF score=0.5359 | retrieval rank=39


**Питання:** навіщо ділити на два етапи, якщо можна ранжувати NCF одразу всі книги?

Ранжувати NCF одразу для всіх книг займе багато часу та ресурсів (особливо на реальних даних). Тому спочатку Two Tower робить ембедінг всіх книг, а потім за допомогою NCF ранжуємо 100 кандидатів для більш точного прогнозу.

---
## Завдання 5. Теоретичний блок (письмові відповіді)

Спираючись на лекцію та на те, що Ви щойно побачили на реальних даних, дайте розгорнуті відповіді в markdown-клітинці нижче.

1. **Чому Recall@10 такий низький?** На реальних даних усі моделі цього ДЗ дають скромний Recall@10. Назвіть щонайменше дві причини (підказки: бідні контентні ознаки — лише 12 жанрів; розрідженість; те, що val-лайки не охоплюють усіх книг, які користувач *міг би* вподобати).
2. **Як покращити якість, не змінюючи архітектуру?** Які додаткові ознаки книг і користувачів з Goodbooks можна було б під'єднати? (автор, рік, середній рейтинг, повний набір тегів через TF-IDF, текстові ембединги опису через BERT...)
3. **Diversity.** Якщо користувач любить фентезі, чому не варто показувати йому 10 фентезі-книг підряд? Як технічно підмішати різноманітність?
4. **Freshness / cold start.** Нова книга має 0 оцінок. Який підхід цього ДЗ зможе рекомендувати її одразу, а який — ні? Чому?
5. **Watch time > CTR (з лекції).** Поясніть, чому YouTube оптимізує час перегляду, а не CTR, і як це технічно вшито у weighted logistic regression.


1. Усі книги описуються тільки 12-ма жанрами, тобто дві книги можуть мати однаковий вектор, але бути різними для читача, оскільки не враховано "піджанр". Немає collaborative information, тобто не враховує того факту, що користувачі, які люблять книгу А, часто люблять також книгу Б.
2. Для книг можна використати додаткові ознаки:
    - автор
    - рік публікації
    - середній рейтинг Goodreads
    - кількість оцінок і відгуків
    - текстовий опис книги
    - ембединги опису, отримані з допомогою BERT
Для користувачів:
    - розподіл оцінок за жанрами
    - улюблених авторів
    - розподіл оцінок за жанрами
    - історій взаємодій
3. Користувачу не варто показувати тільки те, що він уже вподобав, оскільки користувачі також потребують чогось нового, що могло б йому сподобатись і внести різноманіття (нівелювати ефект інформаційної бульбашки). Технічно користувачу поряд із рекомендаціями можна додавати щось із переліку новинок, які набирають популярність.
4. Collaborative підходи залежать від оцінок користувачів, для нових книг відсутня історія взаємодії, модель не знає кому книга може сподобатись. VSM може рекомендувати нову книгу одразу як і Two-Tower якщо відомі її ознаки.
5. Сам факт того, що користувач клікнув на рекомендоване відео - не свідчить про корисність для нього, тут краще підходить показник Watch time (якщо користувачу цікаво - він переглядає відео довше). Технічно це реалізується через weighted logistic regression. Позитивні приклади з довгим переглядом отримують більшу вагу під час навчання, а короткі перегляди - меншу.